# 1D J1-J2=0.0-J3=0.5: PoincareGRU

This notebook is part of the work https://arxiv.org/abs/2604.24337. 
- Here we trained PoincareGRU at `r_max=0.7' with and without the temperature scaling tau in the `sample' method. The results do not differ much qualitatively but without `tau' it is slightly better. 
- When `r_max=1.0', numerical explosion/instability happened and the network did not learn at all. 

In [1]:
import os
import sys
import time
sys.path.append('../utility')
#from j1j2j3_hyprnn_train_loop import *
from j1j2j3_tau_hyprnn_train_loop import *

GPU Available:  False


In [2]:
def set_cpu_deterministic(seed=111):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

set_cpu_deterministic(111)

In [2]:
E_exact = -53.99140745384453
syssize = 100
nssamples = 80
J1 = 1.0
J2 = 0.0
J3 = 0.5
nsteps = 1201
var_tol = 20.0

# With tau in sample generation

In [4]:
# single clamp in hGRU forward pass
r_max=0.7
units=70
fname=f'1d_J1J2J3_results_N={syssize}_tau/hGRU_rmax={r_max}/J2={J2}_J3={J3}'
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=r_max, seed=111)
for name, param in hGRU.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")
print(f'Total params = {sum(p.numel() for p in hGRU.model.parameters())}')

t0=time.time()
mhE, vhE = run_J1J2J3(hGRU, nsteps, syssize, var_tol, J1_=J1, J2_=J2, J3_=J3, Marshall_sign=True, 
                   numsamples=nssamples, lr1=5e-3, lr2=1e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

Layer: rnn.Wz | Size: torch.Size([70, 70])
Layer: rnn.Uz | Size: torch.Size([2, 70])
Layer: rnn.bz | Size: torch.Size([1, 70])
Layer: rnn.Wr | Size: torch.Size([70, 70])
Layer: rnn.Ur | Size: torch.Size([2, 70])
Layer: rnn.br | Size: torch.Size([1, 70])
Layer: rnn.Wh | Size: torch.Size([70, 70])
Layer: rnn.Uh | Size: torch.Size([2, 70])
Layer: rnn.bh | Size: torch.Size([1, 70])
Layer: dense_a.weight | Size: torch.Size([2, 70])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 70])
Layer: dense_p.bias | Size: torch.Size([2])
Total params = 15614


/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1052: ComplexWarning: Casting complex values to real discards the imaginary part
  current = float(metrics)


step: 0, loss: -1.36743, mean energy: 36.40681-0.01641j, varE: 0.83201| Max Radius: 0.7553 | Hyp LR: 1.00e-03| LR: 5.00e-03|tau=1.05
Best model saved at epoch 1 with best E=34.21230+0.02249j, varE=5.97452
Best model saved at epoch 2 with best E=31.04191-0.13232j, varE=15.28242
step: 10, loss: -7.98380, mean energy: 3.02171+0.06168j, varE: 28.03116| Max Radius: 0.7299 | Hyp LR: 1.00e-03| LR: 5.00e-03|tau=1.0490000000000002
step: 20, loss: 6.74677, mean energy: -2.51889+0.03290j, varE: 26.34055| Max Radius: 0.7435 | Hyp LR: 1.00e-03| LR: 5.00e-03|tau=1.048
step: 30, loss: -7.99658, mean energy: -7.43132-0.17595j, varE: 51.03311| Max Radius: 0.7356 | Hyp LR: 1.00e-03| LR: 5.00e-03|tau=1.0470000000000002
step: 40, loss: -26.28845, mean energy: -16.82940-1.26860j, varE: 63.53419| Max Radius: 0.7586 | Hyp LR: 1.00e-03| LR: 5.00e-03|tau=1.046
step: 50, loss: -37.73206, mean energy: -25.00188-1.03184j, varE: 50.79551| Max Radius: 0.8114 | Hyp LR: 1.00e-03| LR: 5.00e-03|tau=1.0450000000000002
s

# No tau in sample generation

## rmax=0.7, units=70

In [10]:
#double clamps in forward pass (by default)
r_max=0.7
units=70
fname=f'1d_J1J2J3_results_N={syssize}/hGRU_rmax={r_max}/J2={J2}_J3={J3}'
hGRU = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=r_max, seed=111)
for name, param in hGRU.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")
print(f'Total params = {sum(p.numel() for p in hGRU.model.parameters())}')

t0=time.time()
mhE, vhE = run_J1J2J3(hGRU, nsteps, syssize, var_tol, J1_=J1, J2_=J2, J3_=J3, Marshall_sign=True, 
                   numsamples=nssamples, lr1=5e-3, lr2=1e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

Layer: rnn.Wz | Size: torch.Size([70, 70])
Layer: rnn.Uz | Size: torch.Size([2, 70])
Layer: rnn.bz | Size: torch.Size([1, 70])
Layer: rnn.Wr | Size: torch.Size([70, 70])
Layer: rnn.Ur | Size: torch.Size([2, 70])
Layer: rnn.br | Size: torch.Size([1, 70])
Layer: rnn.Wh | Size: torch.Size([70, 70])
Layer: rnn.Uh | Size: torch.Size([2, 70])
Layer: rnn.bh | Size: torch.Size([1, 70])
Layer: dense_a.weight | Size: torch.Size([2, 70])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 70])
Layer: dense_p.bias | Size: torch.Size([2])
Total params = 15614


/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1052: ComplexWarning: Casting complex values to real discards the imaginary part
  current = float(metrics)


step: 0, loss: -1.45850, mean energy: 36.40498-0.01128j, varE: 0.85780| Max Radius: 0.7553 | Hyp LR: 1.00e-03| LR: 5.00e-03
Best model saved at epoch 1 with best E=34.19921+0.00024j, varE=6.17862
Best model saved at epoch 2 with best E=31.08487-0.15801j, varE=14.80674
step: 10, loss: -19.82922, mean energy: 13.38517-0.20176j, varE: 48.82125| Max Radius: 0.7517 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 20, loss: -4.09988, mean energy: -2.27269-0.05468j, varE: 31.29982| Max Radius: 0.7113 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 30, loss: -6.04376, mean energy: -8.54478-0.22538j, varE: 51.03996| Max Radius: 0.7228 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 40, loss: -17.16614, mean energy: -19.56855-0.13472j, varE: 57.71149| Max Radius: 0.7632 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 50, loss: -25.38647, mean energy: -30.80551+0.00788j, varE: 41.75265| Max Radius: 0.7843 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 60, loss: 12.45679, mean energy: -38.62407-0.06105j, varE: 32.78785| Max Radius: 0.6426 | 

## No r_max (effectively r_max=1.0)

In [10]:
# With no norm_clamp at all (these files have been overwritten)
t0=time.time()
mhE, vhE = run_J1J2J3(hGRU, nsteps, syssize, var_tol, J1_=J1, J2_=J2, J3_=J3, Marshall_sign=True, 
                   numsamples=nssamples, lr1=5e-3, lr2=1e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

step: 0, loss: -1.87793, mean energy: 36.30353-0.11896j, varE: 1.38249| Max Radius: 0.9497 | Hyp LR: 1.00e-03| LR: 5.00e-03
Best model saved at epoch 1 with best E=31.51367-0.35867j, varE=12.09536
step: 10, loss: 127.29095, mean energy: 14.81566+0.32084j, varE: 86.90134| Max Radius: 1.0000 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 20, loss: 58.77493, mean energy: 8.68996+0.06887j, varE: 58.12153| Max Radius: 1.0000 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 30, loss: 11.49249, mean energy: 12.52981-0.38414j, varE: 60.70670| Max Radius: 0.9991 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 40, loss: -50.21481, mean energy: 6.27906-0.31692j, varE: 37.34361| Max Radius: 0.9953 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 50, loss: -0.47089, mean energy: 2.61104+0.56391j, varE: 33.38179| Max Radius: 0.9656 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 60, loss: -2.44434, mean energy: 34.52995-0.08605j, varE: 1.56106| Max Radius: 1.0000 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 70, loss: -0.03317, mean energy: 33.85313+0